In [ ]:
# x-v_x phase-space animation helpers

import glob
import re
from pathlib import Path
from typing import List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import Video, display
from tqdm import tqdm

PATTERN = "PIC_Part_*.csv"
OUT_MP4 = "xvx_animation.mp4"
OUT_GIF = "xvx_animation.gif"
FPS = 10
SCATTER_SIZE = 2
SCATTER_ALPHA = 0.7
DROP_FRAC = 0.8  # Randomly drop this fraction of particles (keep 1 - DROP_FRAC).
SUBSAMPLE_SEED = 0
USE_VELOCITY = False  # If True, divide momentum by estimated mass per frame.
PHASE_DENSITY_BINS = 256


def extract_step(path: Path) -> int:
    m = re.search(r"PIC_Part_(\d+)\.csv$", path.name)
    return int(m.group(1)) if m else -1


def vx_from_frame(df: pd.DataFrame) -> np.ndarray:
    x = df["X"].to_numpy()
    y = df["Y"].to_numpy()
    vx = df["momentum_0"].to_numpy()
    if USE_VELOCITY:
        m = (x.max() - x.min()) * (y.max() - y.min()) / len(x)
        vx = vx / m
    return vx


def list_snapshot_files(pattern: str = PATTERN) -> Tuple[Path, List[Path]]:
    files = sorted(glob.glob(pattern), key=lambda f: extract_step(Path(f)))
    if not files:
        raise FileNotFoundError(f"No files matching pattern: {pattern}")
    if len(files) < 2:
        raise ValueError("Need at least two CSV files: init (step 1) plus later frames.")
    paths = [Path(f) for f in files]
    return paths[0], paths[1:]


def classify_beams(init_file: Path) -> Tuple[np.ndarray, np.ndarray]:
    required = {"id", "X", "Y", "momentum_0"}
    df_init = pd.read_csv(init_file, usecols=list(required))
    if not required.issubset(df_init.columns):
        raise ValueError(f"{init_file} missing required columns.")

    vx_init = vx_from_frame(df_init)
    left_ids = df_init.loc[vx_init < 0, "id"].to_numpy()
    right_ids = df_init.loc[vx_init >= 0, "id"].to_numpy()
    print(f"Init file: {init_file.name}")
    print(f"Left-moving ids (vx < 0): {len(left_ids)}")
    print(f"Right-moving ids (vx >= 0): {len(right_ids)}")
    return left_ids, right_ids


def subsample_ids(left_ids: np.ndarray, right_ids: np.ndarray,
                  drop_frac: float = DROP_FRAC,
                  seed: int = SUBSAMPLE_SEED) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    all_ids = np.unique(np.concatenate([left_ids, right_ids]))
    n_keep = max(1, int(round((1.0 - drop_frac) * len(all_ids))))
    rng = np.random.default_rng(seed)
    keep_ids = rng.choice(all_ids, size=n_keep, replace=False)
    left_kept = left_ids[np.isin(left_ids, keep_ids)]
    right_kept = right_ids[np.isin(right_ids, keep_ids)]
    print(
        f"Subsample: keep {len(keep_ids)} / {len(all_ids)} particles "
        f"({100 * (1 - drop_frac):.0f}% kept, drop {100 * drop_frac:.0f}%)"
    )
    return keep_ids, left_kept, right_kept


def load_animation_frames(anim_files: List[Path]) -> List[pd.DataFrame]:
    required = {"id", "X", "Y", "momentum_0"}
    frames = []
    for f in tqdm(anim_files, desc="Reading CSV frames"):
        df = pd.read_csv(f, usecols=list(required))
        if not required.issubset(df.columns):
            raise ValueError(f"{f} missing columns. Found: {list(df.columns)}")
        frames.append(df)
    return frames


def compute_axis_limits(frames: List[pd.DataFrame]) -> Tuple[Tuple[float, float], Tuple[float, float]]:
    all_x = np.concatenate([df["X"].to_numpy() for df in frames])
    all_vx = np.concatenate([vx_from_frame(df) for df in frames])
    pad_x = (all_x.max() - all_x.min()) * 0.02 if np.ptp(all_x) != 0 else 1.0
    pad_v = (all_vx.max() - all_vx.min()) * 0.02 if np.ptp(all_vx) != 0 else 1.0
    xlim = (all_x.min() - pad_x, all_x.max() + pad_x)
    vxlim = (all_vx.min() - pad_v, all_vx.max() + pad_v)
    return xlim, vxlim


def compute_phase_space_bin_counts(
    x: np.ndarray, vx: np.ndarray, x_edges: np.ndarray, vx_edges: np.ndarray
) -> np.ndarray:
    hist, _, _ = np.histogram2d(x, vx, bins=(x_edges, vx_edges))
    x_idx = np.clip(np.searchsorted(x_edges, x, side="right") - 1, 0, len(x_edges) - 2)
    vx_idx = np.clip(np.searchsorted(vx_edges, vx, side="right") - 1, 0, len(vx_edges) - 2)
    return hist[x_idx, vx_idx]


def prepare_phase_space_density(
    frames: List[pd.DataFrame], keep_ids: np.ndarray, num_bins: int = PHASE_DENSITY_BINS
) -> Tuple[List[pd.DataFrame], Tuple[float, float]]:
    all_x = np.concatenate([df["X"].to_numpy() for df in frames])
    all_vx = np.concatenate([vx_from_frame(df) for df in frames])

    x_min = float(all_x.min())
    x_max = float(all_x.max())
    vx_min = float(all_vx.min())
    vx_max = float(all_vx.max())
    if x_min == x_max:
        x_min -= 1.0
        x_max += 1.0
    if vx_min == vx_max:
        vx_min -= 1.0
        vx_max += 1.0

    x_edges = np.linspace(x_min, x_max, num_bins + 1)
    vx_edges = np.linspace(vx_min, vx_max, num_bins + 1)

    filtered_frames = []
    rho_max = 1.0
    for df in tqdm(frames, desc="Binning phase-space density"):
        x = df["X"].to_numpy()
        vx = vx_from_frame(df)
        rho_bin = compute_phase_space_bin_counts(x, vx, x_edges, vx_edges)

        keep_mask = df["id"].isin(keep_ids).to_numpy()
        filtered = df.loc[keep_mask].copy()
        filtered["rho_bin"] = rho_bin[keep_mask]
        filtered_frames.append(filtered)
        if len(filtered) > 0:
            rho_max = max(rho_max, float(filtered["rho_bin"].max()))

    return filtered_frames, (1.0, rho_max)


def make_phase_space_figure(xlim, vxlim, rho_lim):
    ylabel = "v_x" if USE_VELOCITY else "p_x (momentum_0)"
    fig, ax = plt.subplots(figsize=(8, 5))
    sc = ax.scatter(
        [], [], s=SCATTER_SIZE, alpha=SCATTER_ALPHA, c=[], cmap="inferno",
        vmin=rho_lim[0], vmax=rho_lim[1], edgecolors="none"
    )
    ax.set_xlim(*xlim)
    ax.set_ylim(*vxlim)
    ax.set_xlabel("x")
    ax.set_ylabel(ylabel)
    title = ax.set_title("")
    ax.grid(True, alpha=0.3)
    cbar = fig.colorbar(sc, ax=ax)
    cbar.set_label("particle count per phase-space bin")
    return fig, ax, sc, title


def build_animation(fig, sc, title, frames, anim_files):
    pbar = tqdm(total=len(frames), desc="Rendering frames", leave=False)

    def init():
        sc.set_offsets(np.empty((0, 2)))
        sc.set_array(np.array([]))
        title.set_text("")
        return sc, title

    def update(i):
        df = frames[i]
        x = df["X"].to_numpy()
        vx = vx_from_frame(df)
        rho = df["rho_bin"].to_numpy()

        sc.set_offsets(np.column_stack([x, vx]))
        sc.set_array(rho)

        step = extract_step(anim_files[i])
        title.set_text(f"x-v_x  |  step {step}  |  frame {i + 1}/{len(frames)}")
        pbar.update(1)
        return sc, title

    anim = animation.FuncAnimation(
        fig, update, frames=len(frames), init_func=init,
        blit=True, interval=100, repeat=False,
    )
    anim._xvx_pbar = pbar
    return anim


def save_animation(anim, fig, out_mp4: str = OUT_MP4, out_gif: str = OUT_GIF, fps: int = FPS) -> str:
    pbar = getattr(anim, "_xvx_pbar", None)
    saved_path = None
    try:
        writer = animation.FFMpegWriter(
            fps=fps, metadata=dict(artist="generated"), bitrate=2000
        )
        anim.save(out_mp4, writer=writer, dpi=150)
        saved_path = out_mp4
    except Exception:
        try:
            anim.save(out_gif, writer="pillow", fps=fps, dpi=150)
            saved_path = out_gif
        except Exception as e:
            if pbar is not None:
                pbar.close()
            plt.close(fig)
            raise RuntimeError(
                "Failed to save animation as MP4 or GIF. Ensure ffmpeg or pillow is available."
            ) from e
    finally:
        if pbar is not None:
            pbar.close()
    return saved_path


In [ ]:
# Run animation pipeline

init_file, anim_files = list_snapshot_files()
anim_files = anim_files
left_ids, right_ids = classify_beams(init_file)
keep_ids, left_ids, right_ids = subsample_ids(left_ids, right_ids)
frames = load_animation_frames(anim_files)
xlim, vxlim = compute_axis_limits(frames)
frames, rho_lim = prepare_phase_space_density(frames, keep_ids)

fig, ax, sc, title = make_phase_space_figure(xlim, vxlim, rho_lim)
anim = build_animation(fig, sc, title, frames, anim_files)

saved_path = save_animation(anim, fig)
plt.close(fig)
display(Video(saved_path, embed=True))
print(f"Saved animation to: {saved_path}")
